# Week 4 Assignment: Exploring Climate Governance Data with Statistics

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

## Objective
Use the ND-GAIN Country Index dataset from the climate-governance project to calculate descriptive statistics for 2–4 numerical variables, examine covariance and correlation between two variables, create a plot, and explain the findings.

**Important:** This is an educational analysis. Correlation does not prove causation, and results should support—not replace—careful climate-policy decisions.

## 1. Import libraries
Pandas is used for data handling, NumPy for numerical calculations, and Matplotlib for visualisation.

In [ ]:
import os
import glob
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## 2. Load the ND-GAIN dataset
This cell attempts to download the same ND-GAIN archive referenced in the project notebook. If the official file URL changes or the download is unavailable, upload the relevant ND-GAIN CSV file to the notebook session and load it manually.

In [ ]:
DATA_URL = 'https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip'
ZIP_PATH = 'ndgain_countryindex_2026.zip'
EXTRACT_DIR = 'ndgain_data'

if not os.path.exists(ZIP_PATH):
    try:
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print('ND-GAIN archive downloaded.')
    except Exception as error:
        print('Automatic download failed:', error)
        print('Upload the ND-GAIN CSV or ZIP file, then use the manual loading option below.')

if os.path.exists(ZIP_PATH):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        archive.extractall(EXTRACT_DIR)
    csv_files = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)
else:
    csv_files = glob.glob('*.csv')

print('CSV files found:', len(csv_files))
for i, path in enumerate(csv_files):
    print(i, path)

## 3. Read and inspect the available CSV files
The code loads available CSV files and displays their names and columns. It then selects a table that appears to contain the Country Index. If automatic selection does not identify the correct table, choose the appropriate CSV from the printed list.

In [ ]:
tables = {}
for path in csv_files:
    try:
        key = os.path.splitext(os.path.basename(path))[0]
        tables[key] = pd.read_csv(path)
        print(f'{key}: shape={tables[key].shape}; columns={list(tables[key].columns[:15])}')
    except Exception as error:
        print('Could not read', path, error)

if not tables:
    raise FileNotFoundError('No CSV files found. Upload the ND-GAIN CSV/ZIP file and rerun this section.')

def score_table_score(df):
    cols = ' '.join(str(c).lower() for c in df.columns)
    return int('vulnerability' in cols) + int('readiness' in cols) + int('country' in cols) + int('year' in cols)

table_name = max(tables, key=lambda name: score_table_score(tables[name]))
df = tables[table_name].copy()
print('\nSelected table:', table_name)
print('Dataset shape:', df.shape)
display(df.head())

## 4. Select 2–4 numerical variables
The analysis prioritises climate vulnerability and readiness scores, then uses other numeric indicators if available. The exact column names can vary by ND-GAIN release, so this cell checks the columns present in the selected table.

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()
print('Numeric columns available:')
print(numeric_columns)

def find_matching_column(columns, phrases):
    for phrase in phrases:
        for column in columns:
            if phrase in str(column).lower():
                return column
    return None

vulnerability_col = find_matching_column(numeric_columns, ['vulnerability'])
readiness_col = find_matching_column(numeric_columns, ['readiness'])
preferred = [c for c in [vulnerability_col, readiness_col] if c is not None]
for c in numeric_columns:
    if c not in preferred and len(preferred) < 4:
        preferred.append(c)

# Use these 2–4 numeric variables for the assignment.
selected_variables = preferred[:4]
if len(selected_variables) < 2:
    raise ValueError('At least two numeric columns are needed. Select a CSV containing numeric ND-GAIN indicators.')

analysis_data = df[selected_variables].copy()
for column in selected_variables:
    analysis_data[column] = pd.to_numeric(analysis_data[column], errors='coerce')
    analysis_data[column] = analysis_data[column].replace([np.inf, -np.inf], np.nan)

print('Selected variables:', selected_variables)
display(analysis_data.head())
display(analysis_data.isna().sum().to_frame('Missing values'))

## 5. Calculate descriptive statistics
For each selected variable, calculate the mean, median, range, variance, and standard deviation. Missing values are excluded. Pandas calculates sample variance and sample standard deviation by default.

In [ ]:
summary = pd.DataFrame(index=selected_variables)
summary['Mean'] = analysis_data.mean()
summary['Median'] = analysis_data.median()
summary['Minimum'] = analysis_data.min()
summary['Maximum'] = analysis_data.max()
summary['Range'] = summary['Maximum'] - summary['Minimum']
summary['Variance'] = analysis_data.var()
summary['Standard deviation'] = analysis_data.std()
display(summary)

### What do the statistics tell us?
- **Mean:** the average value for each indicator.
- **Median:** the middle value when observations are ordered.
- **Range:** the difference between the maximum and minimum values; it shows the spread from the lowest to highest observation.
- **Variance and standard deviation:** describe how much the observations vary around their mean. Standard deviation is expressed in the variable's original units.

Use the table to identify which variable has the largest range or standard deviation. A larger spread means countries or observations differ more on that indicator in this dataset; it does not explain the reason for those differences.

## 6. Calculate covariance and correlation
The first two selected variables are compared. The calculation uses only rows where both variables have observed values.

In [ ]:
var_x, var_y = selected_variables[0], selected_variables[1]
paired_data = analysis_data[[var_x, var_y]].dropna()

covariance = paired_data[var_x].cov(paired_data[var_y])
correlation = paired_data[var_x].corr(paired_data[var_y])

print(f'Variables compared: {var_x} and {var_y}')
print(f'Complete pairs: {len(paired_data)}')
print(f'Covariance: {covariance:.4f}')
print(f'Correlation: {correlation:.4f}')

### What does correlation tell us?
Correlation ranges from **-1 to +1**. A positive correlation means higher values of one variable tend to occur with higher values of the other. A negative correlation means higher values of one tend to occur with lower values of the other. A correlation near zero indicates a weak linear relationship.

Covariance also indicates whether the variables tend to move in the same or opposite directions, but its magnitude depends on the variables' scales. Correlation is standardised and therefore easier to compare.

**Correlation does not necessarily mean causation.** A relationship between climate vulnerability and readiness does not prove that one directly causes the other. Other social, economic, geographic, and environmental factors may be involved, and some ND-GAIN indicators may be related by how the index is constructed.

## 7. Create a plot showing the relationship
A scatter plot displays each observation for the two selected variables.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(paired_data[var_x], paired_data[var_y], alpha=0.7)
plt.xlabel(var_x)
plt.ylabel(var_y)
plt.title(f'{var_x} vs {var_y} (correlation = {correlation:.3f})')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### Interpretation of the plot
An upward pattern suggests a positive association, while a downward pattern suggests a negative association. If points are widely scattered, the linear relationship may be weak. Compare the pattern with the calculated correlation coefficient. Do not claim that one indicator causes another based only on this plot.

## 8. Input → Function/Model → Output
For this project, the input can be a country's governance, economic, and social indicators. A machine-learning model processes those inputs and estimates climate vulnerability as the output.

**Mathematical representation:**

`Predicted climate vulnerability = f(governance, economic, social and contextual indicators)`

Here, `f` represents a model learned from data. The prediction is an estimate for decision support, not a guarantee or a substitute for expert assessment.

## 9. Conclusion
This notebook applies descriptive statistics to numerical indicators from the ND-GAIN climate-governance dataset. Mean and median describe the centre of the data, while range, variance, and standard deviation describe variation. Covariance and correlation show how two selected indicators vary together, and the scatter plot provides a visual check of that relationship. Correlation does not prove causation. These results can help explore climate vulnerability and adaptation readiness, but they should be interpreted alongside contextual evidence and should not be treated as definitive policy conclusions.

## References
1. Notre Dame Global Adaptation Initiative (ND-GAIN), *Country Index — Download Data*: https://gain.nd.edu/our-work/country-index/download-data/
2. Notre Dame Global Adaptation Initiative (ND-GAIN), *Country Index Methodology*: https://gain.nd.edu/our-work/country-index/methodology/